In [43]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
from google.colab.patches import cv2_imshow
from tensorflow.keras import backend as K

In [44]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [45]:
!unzip gdrive/My\ Drive/dataset/midv_2020.zip

Archive:  gdrive/My Drive/dataset/midv_2020.zip
replace MIDV2020/dataset/clips.tar? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [55]:
import json

# reads the json file with the annotations of id cards
def readAnnotationsFile(jsonFile):
  try:
      with open(jsonFile, "r") as read_file:
        data = json.load(read_file)
        return data
  except FileNotFoundError:
    print(f"No such file or directory exists: '{jsonFile}'")
    return None

# retrieves the image name, and the coordinates of id card position in the image
def retrieveImageAttributes(jsonFile):

    # data contains the json data read from the json file
    data = readAnnotationsFile(jsonFile)

    if(data):
        imageAttributesList = []                            # list to contain attributes of all images in the json file
        imageMetadata = data["_via_img_metadata"]           # metadata of image in which the coordinates are present
        imageIdList = data["_via_image_id_list"]            # names of images in the metadata

        for img_id in imageIdList: # Changed 'img' to 'img_id' to avoid confusion with the dictionary 'img' below

            fileName = imageMetadata[img_id]['filename']
            regions = imageMetadata[img_id]['regions']

            # Iterate through regions to find the one with 'all_points_x' (assuming it's the ID card)
            found_region = None
            for region in regions:
                if 'shape_attributes' in region and \
                   'all_points_x' in region['shape_attributes'] and \
                   'all_points_y' in region['shape_attributes']:
                    found_region = region
                    break # Found the region, no need to check further

            if found_region:
                all_points_x = found_region['shape_attributes']['all_points_x']
                all_points_y = found_region['shape_attributes']['all_points_y']
                imageAttributesList.append((fileName, all_points_x, all_points_y))
            else:
                print(f"Warning: No valid polygon region found for image {fileName} in {jsonFile}")

        return imageAttributesList                                             # list of tuples containing (filename, x coordinates and y coordinates) for every image
    else:
        return None

In [56]:
import numpy as np
from skimage.draw import polygon
import cv2 as cv

# creates groud truth mask from all_points_x and all_points_y of four cornres of ID card in the image
def groundTruth(y_len, x_len, row_arr, col_arr, writeFileName):
    img = np.zeros((y_len, x_len), dtype=np.uint8)
    r = np.array(row_arr)
    c = np.array(col_arr)
    rr, cc = polygon(r,c)
    img[rr,cc] = 255

    cv.imwrite(writeFileName, img)

In [57]:
import os
import numpy as np

# creates a new directory
def createDirectory(filePath, newDirectory):
    path = os.path.join(filePath, newDirectory)
    os.makedirs(path, exist_ok=True) # Use os.makedirs to create intermediate directories and avoid error if exists

    return path
# creates ground truth mask at the specified path given
def createMask(filePath, fileName, imgHeight, imgWidth, all_points_y, all_points_x):

    # Clip coordinates to be within image bounds
    clipped_all_points_x = np.clip(all_points_x, 0, imgWidth - 1)
    clipped_all_points_y = np.clip(all_points_y, 0, imgHeight - 1)

    writeFilePath = os.path.join(filePath, fileName)
    groundTruth(imgHeight, imgWidth, clipped_all_points_y, clipped_all_points_x, writeFilePath)

In [58]:
PHOTO_HEIGHT = 4032                     # height of images in photos folder
PHOTO_WIDTH = 2268                      # width of images in photos folder

SCAN_HEIGHT = 3507                      # height of images in scanned_rotated and scanned_upright folder
SCAN_WIDTH = 2480                       # width of images in  scanned_rotated and scanned_upright folder

PATH_PREFIX = 'MIDV2020/dataset'        # name of the folder containing the dataset

pathSuffix = ["annotations/alb_id.json", "annotations/esp_id.json"]
idClass = ["alb_id", "esp_id"]

typePath = ["photo", "scan_rotated", "scan_upright"]

# Dictionary to store image attributes for each ID class from the templates annotations
annotations_by_id_class = {}
for i in range(len(idClass)):
    json_file_path = os.path.join(PATH_PREFIX, "templates", pathSuffix[i]) # Correct path to template annotations
    imageAttributeList = retrieveImageAttributes(json_file_path)
    if imageAttributeList:
        annotations_by_id_class[idClass[i]] = imageAttributeList
    else:
        print(f"Warning: No annotations found for {idClass[i]} at {json_file_path}")

# Now, iterate through each type of image folder (photo, scan_rotated, scan_upright)
# and create masks using the collected annotations
for fol in typePath: # 'photo', 'scan_rotated', 'scan_upright'
    mask_root_dir = createDirectory(os.path.join(PATH_PREFIX, fol), "mask") # e.g. MIDV2020/dataset/photo/mask

    for cls in idClass: # 'alb_id', 'esp_id'
        if cls in annotations_by_id_class:
            id_mask_dir = createDirectory(mask_root_dir, cls) # e.g. MIDV2020/dataset/photo/mask/alb_id

            image_attributes_list = annotations_by_id_class[cls]
            for img_attrs in image_attributes_list:
                file_name = img_attrs[0]
                all_points_x = img_attrs[1]
                all_points_y = img_attrs[2]

                if(fol == "photo"):
                    createMask(id_mask_dir, file_name, PHOTO_HEIGHT, PHOTO_WIDTH, all_points_y, all_points_x)
                else:
                    createMask(id_mask_dir, file_name, SCAN_HEIGHT, SCAN_WIDTH, all_points_y, all_points_x)
        else:
            print(f"Skipping mask creation for {cls} in {fol} as no annotations were loaded.")

# approx 4 minute running time

Skipping mask creation for alb_id in photo as no annotations were loaded.
Skipping mask creation for esp_id in photo as no annotations were loaded.
Skipping mask creation for alb_id in scan_rotated as no annotations were loaded.
Skipping mask creation for esp_id in scan_rotated as no annotations were loaded.
Skipping mask creation for alb_id in scan_upright as no annotations were loaded.
Skipping mask creation for esp_id in scan_upright as no annotations were loaded.


In [59]:
import shutil
import os

src_path = "tmask"
dst_path = "dataset"

# Remove the destination directory if it already exists
if os.path.exists(dst_path):
    shutil.rmtree(dst_path)

destination = shutil.copytree(src_path, dst_path)

In [60]:
TEMP_PATH_PREFIX = PATH_PREFIX
IMG_PATH_SUFFIX = ["images", "mask"]

This part of the code sets up variables and creates the main data directories:

*   **`imageNumbering = [0, 100, 200, 300, 400 ,500]`**:
    *   This list defines a starting index for numbering images and masks for different categories. Each number represents the starting point for a block of 100 images/masks (e.g., 0-99, 100-199, etc.).

*   **`FINAL_DATA_FOLDER = "data"`**:
    *   This variable stores the name of the main directory where all processed images and masks will be stored.

*   **`FINAL_IMG_PATH = os.path.join(FINAL_DATA_FOLDER, IMG_PATH_SUFFIX[0])`**:
    *   Constructs the full path for the directory where processed images will be stored (e.g., `data/images`). `IMG_PATH_SUFFIX[0]` is assumed to be "images".

*   **`FINAL_MASK_PATH = os.path.join(FINAL_DATA_FOLDER, IMG_PATH_SUFFIX[1])`**:
    *   Constructs the full path for the directory where processed masks will be stored (e.g., `data/mask`). `IMG_PATH_SUFFIX[1]` is assumed to be "mask".

*   **`os.mkdir(FINAL_DATA_FOLDER)`**:
    *   Creates the main `data` directory. If it already exists, this would raise a `FileExistsError` (which was addressed in previous steps by adding `shutil.rmtree` checks).

*   **`os.mkdir(FINAL_IMG_PATH)`**:
    *   Creates the `images` subdirectory within the `data` folder.

*   **`os.mkdir(FINAL_MASK_PATH)`**:
    *   Creates the `mask` subdirectory within the `data` folder.

*   **`imgNum = 0`**:
    *   This variable acts as a counter, likely to keep track of the current block of image numbers being processed, and is incremented in later parts of the full loop.

In [61]:
shutil.rmtree("dataset")

In [64]:
import shutil
import os

imageNumbering = [0, 100, 200, 300, 400 ,500]

FINAL_DATA_FOLDER = "data"
FINAL_IMG_PATH = os.path.join(FINAL_DATA_FOLDER, "images")
FINAL_MASK_PATH = os.path.join(FINAL_DATA_FOLDER, "mask")

# Remove existing data directory if it exists
if os.path.exists(FINAL_DATA_FOLDER):
    shutil.rmtree(FINAL_DATA_FOLDER)

os.mkdir(FINAL_DATA_FOLDER)
os.mkdir(FINAL_IMG_PATH)
os.mkdir(FINAL_MASK_PATH)

imgNum = 0
for path_type_idx, path_type in enumerate(typePath):

  # Process images for the current path_type
  current_dest_path_images = FINAL_IMG_PATH
  # The original images are likely directly under MIDV2020/dataset/typePath/idClass
  # No 'images' subdirectory in the source path for original images

  for cls in idClass:
    source_dir_path = os.path.join(TEMP_PATH_PREFIX, path_type, cls) # Corrected source path for original images

    if os.path.exists(source_dir_path):
      files_in_dir = os.listdir(source_dir_path)
      if len(files_in_dir) == 100:
        tmp_base_num = imageNumbering[imgNum + (0 if cls == "alb_id" else 3)]
        for i in range(0,100):
          fileName = f"{i:02}.jpg"
          newName = f"{(tmp_base_num+i):03}.jpg"
          source_file = os.path.join(source_dir_path, fileName)
          destination_file = os.path.join(current_dest_path_images, newName)

          if os.path.exists(source_file):
              os.rename(source_file, destination_file)
          else:
              print(f"Warning: Source image file not found: {source_file}")
      else:
        print(f"Warning: Expected 100 image files in {source_dir_path}, but found {len(files_in_dir)} files. Skipping.")
    else:
      print(f"Warning: Source image directory not found: {source_dir_path}. Skipping.")

  # Process masks next
  current_dest_path_masks = FINAL_MASK_PATH
  folder_suffix_masks = IMG_PATH_SUFFIX[1] # 'mask' - This is still needed for mask source paths

  for cls in idClass:
    source_dir_path = os.path.join(TEMP_PATH_PREFIX, path_type, folder_suffix_masks, cls) # Path for generated masks

    if os.path.exists(source_dir_path):
      files_in_dir = os.listdir(source_dir_path)
      if len(files_in_dir) == 100:
        tmp_base_num = imageNumbering[imgNum + (0 if cls == "alb_id" else 3)]
        for i in range(0,100):
          fileName = f"{i:02}.jpg"
          newName = f"{(tmp_base_num+i):03}.jpg"
          source_file = os.path.join(source_dir_path, fileName)
          destination_file = os.path.join(current_dest_path_masks, newName)

          if os.path.exists(source_file):
              os.rename(source_file, destination_file)
          else:
              print(f"Warning: Source mask file not found: {source_file}")
      else:
        print(f"Warning: Expected 100 mask files in {source_dir_path}, but found {len(files_in_dir)} files. Skipping.")
    else:
      print(f"Warning: Source mask directory not found: {source_dir_path}. Skipping.")

  imgNum += 1

In [63]:
imageNumbering = [0, 100, 200, 300, 400 ,500]

FINAL_DATA_FOLDER = "data"
FINAL_IMG_PATH = os.path.join(FINAL_DATA_FOLDER, IMG_PATH_SUFFIX[0])
FINAL_MASK_PATH = os.path.join(FINAL_DATA_FOLDER, IMG_PATH_SUFFIX[1])
os.mkdir(FINAL_DATA_FOLDER)
os.mkdir(FINAL_IMG_PATH)
os.mkdir(FINAL_MASK_PATH)

imgNum = 0
for path in typePath:
  imgPath = FINAL_IMG_PATH
  for fol in IMG_PATH_SUFFIX:
    for cls in idClass:
      idPath = os.path.join(TEMP_PATH_PREFIX,path,fol,cls)
      if(len(os.listdir(idPath))==100):
        if(cls == "alb_id"):
          tmp_ = imgNum
        else:
          tmp_ = imgNum + 3
        for i in range(0,100):
          fileName = f"{i:02}"+str(".jpg")
          newName = f"{(imageNumbering[tmp_]+i):02}"+str(".jpg")
          os.rename(os.path.join(idPath,fileName),os.path.join(imgPath,newName))
      else:
        print("Files not read proerly")
    imgPath = FINAL_MASK_PATH
  imgNum += 1
  break


FileExistsError: [Errno 17] File exists: 'data'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
imageNames = os.listdir("data/images")
maskNames = os.listdir("data/mask")

In [ ]:
imageNames.sort()

In [ ]:
maskNames.sort()

In [ ]:
for i in range(1,200):
  if(imageNames[i] != maskNames[i]):
    print(imageNames[i])
    print(maskNames[i])
    print("------------------------")


In [ ]:
print(imageNames[81])
print(maskNames[81])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(imageNames, maskNames, test_size=0.2, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42) # 0.25 x 0.8 = 0.2

In [ ]:
print(len(X_train), len(y_train))

In [ ]:
print(len(X_val), len(y_val))

In [ ]:
print(len(X_test), len(y_test))

In [ ]:
X_train[0]

In [ ]:
X_val[0]

In [ ]:
256*2

In [ ]:
def read_images(image_list):
  temp_image_list = []
  for image in image_list:
    orig_img = cv.imread(os.path.join("data/images",image))
    res_img = cv.resize(orig_img, (256,256), interpolation=cv.INTER_NEAREST)/255
    temp_image_list.append(tf.constant(res_img))
  return temp_image_list

def read_masks(mask_list):
  temp_mask_list = []
  for mask in mask_list:
    orig_msk = cv.imread(os.path.join("data/mask",mask),cv.IMREAD_GRAYSCALE)
    res_msk = cv.resize(orig_msk, (256,256), interpolation=cv.INTER_NEAREST)/255
    temp_mask_list.append(tf.constant(np.expand_dims(res_msk,axis=2)))
  return temp_mask_list

In [ ]:
train_images = read_images(X_train)
train_masks = read_masks(y_train)
val_images = read_images(X_val)
val_masks = read_masks(y_val)
test_images = read_images(X_test)
test_masks = read_masks(y_test)

# approx 1m 40s

In [ ]:
len(train_images)

In [ ]:
len(train_masks)

In [ ]:
len(val_images)

In [ ]:
len(val_masks)

In [ ]:
len(test_images)

In [ ]:
len(test_masks)

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_masks))
val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_masks))
test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_masks))

In [ ]:
BATCH_SIZE = 16
train_batches = train_dataset.batch(BATCH_SIZE).repeat()
train_batches = train_batches.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
val_batches = val_dataset.batch(BATCH_SIZE).repeat()
val_batches = val_batches.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

In [ ]:
test_batches = test_dataset.batch(BATCH_SIZE).repeat()
test_batches = test_batches.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

In [ ]:
def display(display_list):
 plt.figure()

 title = ["Input Image", "True Mask", "Predicted Mask"]

 for i in range(len(display_list)):
   plt.subplot(1, len(display_list), i+1)
   plt.title(title[i])
   plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
   plt.axis("off")
 plt.show()

sample_batch = next(iter(train_batches))
random_index = np.random.choice(sample_batch[0].shape[0])
sample_image, sample_mask = sample_batch[0][random_index], sample_batch[1][random_index]
display([sample_image, sample_mask])

In [ ]:
def double_conv_block(x, n_filters):

   # Conv2D then ReLU activation
   x = layers.Conv2D(n_filters, 3, padding = "same", activation = "relu", kernel_initializer = "he_normal")(x)
   # Conv2D then ReLU activation
   x = layers.Conv2D(n_filters, 3, padding = "same", activation = "relu", kernel_initializer = "he_normal")(x)

   return x

In [ ]:
def downsample_block(x, n_filters):
   f = double_conv_block(x, n_filters)
   p = layers.MaxPool2D(2)(f)
   p = layers.Dropout(0.3)(p)

   return f, p

In [ ]:
def upsample_block(x, conv_features, n_filters):
   # upsample
   x = layers.Conv2DTranspose(n_filters, 3, 2, padding="same")(x)
   # concatenate
   x = layers.concatenate([x, conv_features])
   # dropout
   x = layers.Dropout(0.3)(x)
   # Conv2D twice with ReLU activation
   x = double_conv_block(x, n_filters)

   return x

In [ ]:
def build_unet_model():
 # inputs
   inputs = layers.Input(shape=(512,512,3))

   # encoder: contracting path - downsample
   # 1 - downsample
   f1, p1 = downsample_block(inputs, 64)
   # 2 - downsample
   f2, p2 = downsample_block(p1, 128)
   # 3 - downsample
   f3, p3 = downsample_block(p2, 256)
   # 4 - downsample
   f4, p4 = downsample_block(p3, 512)

   # 5 - bottleneck
   bottleneck = double_conv_block(p4, 1024)

   # decoder: expanding path - upsample
   # 6 - upsample
   u6 = upsample_block(bottleneck, f4, 512)
   # 7 - upsample
   u7 = upsample_block(u6, f3, 256)
   # 8 - upsample
   u8 = upsample_block(u7, f2, 128)
   # 9 - upsample
   u9 = upsample_block(u8, f1, 64)

   # outputs
   outputs = layers.Conv2D(3, 1, padding="same", activation = "softmax")(u9)

   # unet model with Keras Functional API
   unet_model = tf.keras.Model(inputs, outputs, name="U-Net")

   return unet_model

In [ ]:
unet_model = build_unet_model()

In [ ]:
unet_model.compile(optimizer=tf.keras.optimizers.Adam(),
                  loss="sparse_categorical_crossentropy",
                  metrics=['accuracy'])

In [ ]:
unet_model.summary()

In [ ]:
len(train_dataset)

In [ ]:
def MyCallback():
  checkpointer = tf.keras.callbacks.ModelCheckpoint(filepath='unet.h5', verbose=0, save_best_only=True, save_weights_only=True)
  callbacks = [checkpointer]
  return callbacks

# # inheritance for training process plot
# class PlotLearning(keras.callbacks.Callback):

#     def on_train_begin(self, logs={}):
#         self.i = 0
#         self.x = []
#         self.losses = []
#         self.val_losses = []
#         self.acc = []
#         self.val_acc = []
#         self.logs = []
#     def on_epoch_end(self, epoch, logs={}):
#         self.logs.append(logs)
#         self.x.append(self.i)
#         self.losses.append(logs.get('loss'))
#         self.val_losses.append(logs.get('val_loss'))
#         self.acc.append(logs.get('mean_iou'))
#         self.val_acc.append(logs.get('val_mean_iou'))
#         self.i += 1
#         print('i=',self.i,'loss=',logs.get('loss'),'val_loss=',logs.get('val_loss'),'mean_iou=',logs.get('mean_iou'),'val_mean_iou=',logs.get('val_mean_iou'))


#         if(logs.get('mean_iou')>0.97):
#           print("\nReached 97% accuracy cancelling training")
#           self.model.stop_training = True

In [ ]:
NUM_EPOCHS = 40

TRAIN_LENGTH = len(train_dataset)
STEPS_PER_EPOCH = TRAIN_LENGTH // BATCH_SIZE

# VAL_SUBSPLITS = 5
TEST_LENTH = len(val_dataset)
# VALIDATION_STEPS = TEST_LENTH // BATCH_SIZE // VAL_SUBSPLITS
VALIDATION_STEPS = TEST_LENTH // BATCH_SIZE

model_history = unet_model.fit(train_batches,
                              epochs=NUM_EPOCHS,
                              steps_per_epoch=STEPS_PER_EPOCH,
                              validation_steps=VALIDATION_STEPS,
                              validation_data=val_batches,
                               callbacks=[MyCallback()])

In [ ]:
temp_test = next(iter(test_dataset))

In [ ]:
PREDICT_LENGTH = len(test_dataset)
PREDICTION_STEPS = PREDICT_LENGTH // BATCH_SIZE

In [ ]:
pred = unet_model.predict(x=test_batches,
                          steps=PREDICTION_STEPS, verbose=0)

In [ ]:
pred.shape

In [ ]:
pred[0].shape

In [ ]:
import matplotlib
from google.colab import files

def pred_masks(predictions, test_batch):
  for batch in test_batch:
    first_batch = batch
    break
  true_images = first_batch[0]
  true_masks = first_batch[1]

  curr_iter = 0
  max_imgs = 20
  all_images = []
  # res_path = createDirectory('/content/','prediction')
  for true_img, true_msk, pred_msk in zip(true_images, true_masks, predictions):
    comparison = []
    img = np.squeeze(true_img).copy()*255
    msk = np.squeeze(true_msk).copy()*255
    msk = np.stack((msk,)*3, axis=-1)
    res = pred_msk.copy()
    res = res[:,:,0]
    res[res >= 0.9] = 255
    res[res < 0.9] = 0
    res = np.stack((res,)*3, axis=-1)
    comparison = np.concatenate((img,msk,res), axis=1)
    cv2_imshow(comparison)

    # temp_comparison = comparison.copy()
    # temp_comparison = temp_comparison.astype(int)
    # temp_comparison = np.uint8(temp_comparison)

    # img_write = os.path.join(res_path,f'{curr_iter}.png')
    # matplotlib.image.imsave(img_write, temp_comparison)
    # files.download(img_write)
    curr_iter += 1
    if curr_iter == 20:
      break

In [ ]:
# shutil.rmtree("prediction")

In [ ]:
pred_masks(pred, test_batches)

In [ ]:
t = write_img.copy()

In [ ]:
t = t.astype(int)
t = np.uint8(t)

In [ ]:
import matplotlib

matplotlib.image.imsave("/content/0.jpg", t)

In [ ]:
from google.colab import files
files.download("/content/0.jpg")

In [ ]:
cv.imwrite("C:/Users/rathi/Desktop/results/0.jpg",write_img)